In [ ]:
import torch
import sys
import os.path as osp
import os
import sys
import numpy as np

sys.path.append("/afs/cern.ch/work/a/adevita/public/Tracking_DC_new")
from src.dataset.dataset import SimpleIterDataset
from src.utils.train_utils import to_filelist
from torch.utils.data import DataLoader
import dgl  # CPU only version for now
from tqdm import tqdm
from torch_scatter import scatter_sum
import matplotlib.pyplot as plt
import pickle
import numpy as np
import mplhep as hep


hep.style.use("CMS")
import matplotlib
matplotlib.rc('font', size=13)

import plotly
import plotly.graph_objects as go
import plotly.offline as pyo
from plotly.subplots import make_subplots


In [ ]:
class Args:
    def __init__(self, datasets):
        self.data_train = [datasets]
        self.data_val = [datasets]
        #self.data_train = files_train
        self.data_config = '/afs/cern.ch/work/a/adevita/public/Tracking_DC_new/config_files/config_tracking_global_vector.yaml'
        self.extra_selection = None
        self.train_val_split = 0.8
        self.data_fraction = 1
        self.file_fraction = 1
        self.fetch_by_files = False
        self.fetch_step = 1
        self.steps_per_epoch = None
        self.in_memory = False
        self.local_rank = None
        self.copy_inputs = False
        self.no_remake_weights = False
        self.batch_size = 2
        self.num_workers = 0
        self.demo = False
        self.laplace = False
        self.diffs = False
        self.class_edges = False

In [ ]:
# This block is the same as 1_dataset.ipynb

#"/eos/experiment/fcc/ee/datasets/DC_tracking/Pythia_evaluation/Zcard/reco_Zcard_1.root"
datasets = {
    "test": "/eos/user/a/adevita/public/saveSpace/testingFolder/flagTests/Pythia/Zcard/Zcard_graphs_42.root",
    "train": "/eos/user/a/adevita/public/saveSpace/testingFolder/flagTests/Pythia/Zcard/Zcard_graphs_42.root"
}

args = {key: Args(value) for key, value in datasets.items()}

datas = {}
files_dict = {}
for key in datasets:
    train_range = (0, args[key].train_val_split)
    train_file_dict, train_files = to_filelist(args[key], 'val')
    
    train_data = SimpleIterDataset(train_file_dict, args[key].data_config, for_training=False,
                                   extra_selection=args[key].extra_selection,
                                   remake_weights=True,
                                   load_range_and_fraction=(train_range, args[key].data_fraction),
                                   file_fraction=args[key].file_fraction,
                                   fetch_by_files=args[key].fetch_by_files,
                                   fetch_step=args[key].fetch_step,
                                   infinity_mode=False,
                                   in_memory=args[key].in_memory,
                                   async_load=False,
                                   name='train')
    datas[key] = train_data
    files_dict[key] = train_files




In [ ]:
itera = iter(train_data)

for i in range(0,2):
    g, y = next(itera)
    
    print("Graph", g)
    print("Y", y)

In [ ]:
good_files = []

In [ ]:
import os
import sys

root_dir = "/eos/experiment/fcc/users/a/adevita/idea_v3_o1_dataset/Pythia/Zcard"

for filename in os.listdir(root_dir):
    if not filename.endswith(".root"):
        continue

    filepath = os.path.join(root_dir, filename)
    
    if filepath in good_files:
        continue
    
    print(f"Testing: {filepath}")
    try:
        args = Args(filepath)
        train_range = (0, args.train_val_split)
        file_dict, _ = to_filelist(args, 'val')

        train_data = SimpleIterDataset(
            file_dict, args.data_config, for_training=False,
            extra_selection=args.extra_selection,
            remake_weights=True,
            load_range_and_fraction=(train_range, args.data_fraction),
            file_fraction=args.file_fraction,
            fetch_by_files=args.fetch_by_files,
            fetch_step=args.fetch_step,
            infinity_mode=False,
            in_memory=args.in_memory,
            async_load=False,
            name='train'
        )
        
        itera = iter(train_data)
        
        good_files.append(filepath)
        
    except RuntimeError as e:
        print(f"Deleting problematic file: {filepath}")
        os.remove(filepath)
            


In [ ]:
import os
import sys

root_dir = "/eos/experiment/fcc/users/a/adevita/idea_v3_o1_dataset/Pythia/Zcard"

for filename in os.listdir(root_dir):
    if not filename.endswith(".root"):
        continue

    filepath = os.path.join(root_dir, filename)
    
    if filepath in good_files:
        continue
    
    print(f"Testing: {filepath}")
    try:
        args = Args(filepath)
        train_range = (0, args.train_val_split)
        file_dict, _ = to_filelist(args, 'val')

        train_data = SimpleIterDataset(
            file_dict, args.data_config, for_training=False,
            extra_selection=args.extra_selection,
            remake_weights=True,
            load_range_and_fraction=(train_range, args.data_fraction),
            file_fraction=args.file_fraction,
            fetch_by_files=args.fetch_by_files,
            fetch_step=args.fetch_step,
            infinity_mode=False,
            in_memory=args.in_memory,
            async_load=False,
            name='train'
        )
        
        itera = iter(train_data)


        while True:
            
            g, y = next(itera)

            def check_tensor(tensor, name):
                if torch.isnan(tensor).any():
                    print(f"NaN in {name}")
                if torch.isinf(tensor).any():
                    print(f"Inf in {name}")

                    
            if isinstance(y, torch.Tensor):
                check_tensor(y, "label y")
                        
            for attr_name in dir(g):
                if attr_name.startswith('_'):
                    continue
                
                attr = getattr(g, attr_name)
                if isinstance(attr, torch.Tensor):
                    check_tensor(attr, f"g.{attr_name}")
                            
        
    
    except StopIteration:
        print("Stop Iteration reached, file is good.")
        good_files.append(filepath)
      
    except Exception as e:
        print(f"Error with file {filepath}: {e}")